# HFCS raw data export

Read the original Stata files for the five HFCS waves and export the fields used in the wealth and inheritance work. Keep the raw files unchanged. Stata missing codes and label dictionaries are stored separately.


## Paths and imports


In [1]:
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Iterator
import pandas as pd
import re
import numbers
from pandas.io.stata import StataReader, StataMissingValue


# Keys are HFCS *survey waves*, not necessarily the final folder's release year.
# For example, the 2023 wave is in the folder ending in STATA_2024.
WAVE_FOLDERS = {
    2010: ("HFCS_UDB_1_5_Stata_2010", ""),
    2014: ("HFCS_UDB_2_5_Stata_2014", "Data"),
    2017: ("HFCS_UDB_3_3_STATA_2017", "Data"),
    2021: ("HFCS_UDB_4_0_Stata_2021", ""),
    2023: ("HFCS_UDB_5_0_STATA_2024", ""),
}

# H: household; HN: household non-core; P: person; PN: person non-core;
# D: derived variables; W: replicate weight file (one per wave).
TABLES = ("h", "hn", "p", "pn", "d", "w")

# Optional: hard-code this only if automatic sibling-folder discovery fails.
# Example: HFCS_DATA_DIR = Path(r"C:\...\RA\HFCS Data Download")
HFCS_DATA_DIR = None


In [2]:
def data_root() -> Path:
    """Find the sibling 'HFCS Data Download' folder from the Jupyter cwd."""
    explicit = HFCS_DATA_DIR or os.environ.get("HFCS_DATA_DIR")
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.is_dir():
            raise FileNotFoundError(f"HFCS data folder not found: {path}")
        return path

    cwd = Path.cwd().resolve()
    # Jupyter may launch from the repo root, HFCS/, or an ancestor.
    for anchor in (cwd, *cwd.parents):
        for candidate in (anchor / "HFCS Data Download", anchor.parent / "HFCS Data Download"):
            if candidate.is_dir():
                return candidate.resolve()
    raise FileNotFoundError(
        "Cannot find 'HFCS Data Download' beside Research-Assistant-Git. "
        "Set HFCS_DATA_DIR in the first cell to its full local path."
    )


def raw_path(wave: int, table: str, imputation: int | None = None) -> Path:
    """Return the original .dta path. Never modifies the source file."""
    if wave not in WAVE_FOLDERS:
        raise ValueError(f"Unknown HFCS wave: {wave}")
    table = table.lower()
    if table not in TABLES:
        raise ValueError(f"Unknown table: {table!r}. Choose from {TABLES}")

    folder, subfolder = WAVE_FOLDERS[wave]
    folder_path = data_root() / folder
    if subfolder:
        folder_path /= subfolder

    if table == "w":
        if imputation is not None:
            raise ValueError("w.dta has no imputation suffix; use imputation=None")
        return folder_path / "w.dta"
    if imputation not in (1, 2, 3, 4, 5):
        raise ValueError("Select imputation 1 to 5 for h/hn/p/pn/d")
    return folder_path / f"{table}{imputation}.dta"

# Data exports stay beside the repository, not inside it.
OUTPUT_DIR = data_root() / "HFCS_selected_raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Input files

The 2014 and 2017 releases have an extra `Data` directory. The 2023 survey uses the folder labelled `STATA_2024`. These differences are handled in `raw_path`.


In [3]:
def file_specs(
    waves: Iterable[int] | None = None,
    tables: Iterable[str] | None = None,
    imputation: int | None = None,
) -> Iterator[tuple[int, str, int | None, Path]]:
    """Yield source file descriptions without loading them."""
    selected_waves = list(WAVE_FOLDERS) if waves is None else list(waves)
    selected_tables = list(TABLES) if tables is None else [t.lower() for t in tables]
    for wave in selected_waves:
        for table in selected_tables:
            if table == "w":
                # Replicate weights are in one wave-level file, not five files.
                if imputation is None:
                    yield wave, table, None, raw_path(wave, table)
            else:
                imps = range(1, 6) if imputation is None else [imputation]
                for imp in imps:
                    yield wave, table, imp, raw_path(wave, table, imp)


specs = list(file_specs())
inventory = pd.DataFrame([
    {
        "wave": wave,
        "table": table,
        "imputation": imp,
        "file": path.name,
        "exists": path.is_file(),
    }
    for wave, table, imp, path in specs
])

print("HFCS root:", data_root())
print(f"Expected raw .dta files: {len(specs)}")
print(f"Found: {int(inventory['exists'].sum())} | Missing: {int((~inventory['exists']).sum())}")
display(inventory.groupby(["wave", "table"], as_index=False)["exists"].agg(["sum", "count"]))
if not inventory["exists"].all():
    display(inventory.loc[~inventory["exists"]])


HFCS root: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\HFCS Data Download
Expected raw .dta files: 130
Found: 130 | Missing: 0


,wave,table,sum,count
0,2010,d,5,5
1,2010,h,5,5
2,2010,hn,5,5
3,2010,p,5,5
4,2010,pn,5,5
5,2010,w,1,1
6,2014,d,5,5
7,2014,h,5,5
8,2014,hn,5,5
9,2014,p,5,5


## Read Stata tables

`load_table` reads the data without converting the Stata category codes or missing-value codes.


In [4]:
def load_table(wave: int, table: str, imputation: int | None = None) -> pd.DataFrame:
    """Read all rows/columns with Stata values and extended missing codes intact."""
    path = raw_path(wave, table, imputation)
    if not path.is_file():
        raise FileNotFoundError(f"Missing HFCS file: {path}")
    return pd.read_stata(
        path,
        convert_missing=True,        # Preserve ., .a, .b, etc.
        convert_categoricals=False,  # Keep numeric codes, not value-label text
        convert_dates=False,         # Keep original Stata date representation
        preserve_dtypes=True,
    )


@dataclass
class RawTable:
    wave: int
    table: str
    imputation: int | None
    path: Path
    data: pd.DataFrame


def iter_raw_tables(
    waves: Iterable[int] | None = None,
    tables: Iterable[str] | None = None,
    imputation: int | None = None,
) -> Iterator[RawTable]:
    """Yield complete source tables one by one; no join or transformation."""
    for wave, table, imp, path in file_specs(waves, tables, imputation):
        yield RawTable(wave, table, imp, path, load_table(wave, table, imp))


In [5]:
# Example: load h1.dta from the 2023 HFCS wave.
SELECTED_WAVE = 2023
SELECTED_TABLE = "h"
SELECTED_IMPUTATION = 1  # Use None for "w"

raw_df = load_table(SELECTED_WAVE, SELECTED_TABLE, SELECTED_IMPUTATION)
print("Source:", raw_path(SELECTED_WAVE, SELECTED_TABLE, SELECTED_IMPUTATION))
print(f"Rows: {len(raw_df):,} | Columns: {len(raw_df.columns):,}")
print("Variable names:", list(raw_df.columns))
# raw_df is untouched and ready for subsequent, separate processing notebooks.
# Do not use raw_df.head() if you plan to commit this notebook with outputs.


C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_13804\1643110559.py:6: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  return pd.read_stata(


Source: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\HFCS Data Download\HFCS_UDB_5_0_STATA_2024\h1.dta
Rows: 84,755 | Columns: 914
Variable names: ['id', 'survey', 'sa0010', 'sa0100', 'im0100', 'hw0010', 'ha0100', 'fha0100', 'hb0100', 'fhb0100', 'hb0100_b', 'fhb0100_b', 'hb0200', 'fhb0200', 'hb0300', 'fhb0300', 'hb0400', 'fhb0400', 'hb0410', 'fhb0410', 'hb0500', 'fhb0500', 'hb0600', 'fhb0600', 'hb0700', 'fhb0700', 'hb0800', 'fhb0800', 'hb0900', 'fhb0900', 'hb1000', 'fhb1000', 'hb1010', 'fhb1010', 'hb1101', 'fhb1101', 'hb1102', 'fhb1102', 'hb1103', 'fhb1103', 'hb1131a', 'fhb1131a', 'hb1131b', 'fhb1131b', 'hb1132a', 'fhb1132a', 'hb1132b', 'fhb1132b', 'hb1133a', 'fhb1133a', 'hb1133b', 'fhb1133b', 'hb1201a', 'fhb1201a', 'hb1201b', 'fhb1201b', 'hb1201c', 'fhb1201c', 'hb1201d', 'fhb1201d', 'hb1202a', 'fhb1202a', 'hb1202b', 'fhb1202b', 'hb1202c', 'fhb1202c', 'hb1202d', 'fhb1202d', 'hb1203a', 'fhb1203a', 'hb1203b', 'fhb1203b', 'hb1203c', 'fhb1203c'

## Full read check

Optional check for all source files. Disabled for a normal export run.


In [6]:
RUN_FULL_READ_AUDIT = False

if RUN_FULL_READ_AUDIT:
    checks = []
    for wave, table, imp, path in file_specs():
        df = load_table(wave, table, imp)
        checks.append({
            "wave": wave,
            "table": table,
            "imputation": imp,
            "file": path.name,
            "rows": len(df),
            "columns": len(df.columns),
        })
        del df
    read_audit = pd.DataFrame(checks)
    display(read_audit)
else:
    print("Full read audit skipped. Set RUN_FULL_READ_AUDIT=True to run it.")


Full read audit skipped. Set RUN_FULL_READ_AUDIT=True to run it.


## Selected variables and Parquet export

Household (`h`), derived (`d`) and person (`p`) files are exported for all five imputations. The added fields cover inheritance response flags, reference-person age and age brackets, and Italy's historical survey weight `hw0010a`. The selection is explicit so additional variables can be added here later.

Exports are written to `HFCS Data Download/HFCS_selected_raw`, outside Git.


In [7]:
# Original field names are kept. Pattern matching is case-insensitive.
VARIABLE_PATTERNS = {
    "h": [
        r"sa\d{4}", r"im0100", r"hw0010a?",
        r"hh0[1-5]\d{2}", r"fhh0[1-5]\d{2}",
        r"hb0600", r"fhb0600", r"hi\d{4}",
    ],
    "d": [
        r"sa\d{4}", r"im0100", r"dn\w+", r"da\w+",
        r"dl\w+", r"di\w+", r"dh\w+",
    ],
    "p": [
        r"sa\d{4}", r"im0100", r"ra0010", r"ra0100",
        r"ra0300(?:_b)?", r"fra0300(?:_b)?",
    ],
}


def select_variables(available, patterns):
    return [name for name in available
            if any(re.fullmatch(pattern, name, flags=re.I) for pattern in patterns)]


def preserve_stata_missing(df):
    """Keep Stata's ordinary and extended missing values in companion columns."""
    df = df.copy()
    missing_columns = []
    for col in list(df.columns):
        s = df[col]
        if s.dtype != object:
            continue
        mask = s.map(lambda x: isinstance(x, StataMissingValue))
        if not mask.any():
            continue

        companion = f"{col}__stata_missing"
        if companion in df.columns:
            raise ValueError(f"Duplicate missing-code column: {companion}")
        codes = pd.Series(pd.NA, index=df.index, dtype="string")
        codes.loc[mask] = s.loc[mask].map(str).tolist()
        values = s.where(~mask, other=None)
        first = next((x for x in values if x is not None and not pd.isna(x)), None)

        if isinstance(first, str):
            df[col] = values.astype("string")
        elif isinstance(first, numbers.Integral):
            df[col] = pd.array(values, dtype="Int64")
        else:
            df[col] = pd.array(values, dtype="Float64")
        df[companion] = codes
        missing_columns.append(col)
    return df, missing_columns


export_rows = []
for wave in WAVE_FOLDERS:
    for imp in range(1, 6):
        for table, patterns in VARIABLE_PATTERNS.items():
            source = raw_path(wave, table, imp)
            with StataReader(source, convert_missing=True,
                             convert_categoricals=False, convert_dates=False,
                             preserve_dtypes=True) as reader:
                selected = select_variables(reader.variable_labels(), patterns)
                if not selected:
                    raise ValueError(f"No matching columns in {source}")
                df = reader.read(columns=selected)

            original_rows = len(df)
            df, missing_columns = preserve_stata_missing(df)
            destination = OUTPUT_DIR / f"hfcs_{wave}_{table}_imp{imp}.parquet"
            df.to_parquet(destination, engine="pyarrow", compression="zstd", index=False)
            check = pd.read_parquet(destination)
            if len(check) != original_rows or not check.columns.equals(df.columns):
                raise AssertionError(f"Export check failed: {destination.name}")
            export_rows.append({
                "wave": wave, "imputation": imp, "table": table,
                "rows": original_rows, "raw_variables": len(selected),
                "stata_missing_columns": len(missing_columns),
                "output": destination.name,
            })
            print(f"{destination.name}: {original_rows:,} rows, {len(selected)} variables")
            del df, check

export_audit = pd.DataFrame(export_rows)
export_audit.to_csv(OUTPUT_DIR / "export_audit.csv", index=False)
print("Exports:", len(export_audit), "| Folder:", OUTPUT_DIR)
display(export_audit.groupby(["wave", "table"], as_index=False)["raw_variables"].min())


hfcs_2010_h_imp1.parquet: 68,627 rows, 36 variables
hfcs_2010_d_imp1.parquet: 68,627 rows, 164 variables
hfcs_2010_p_imp1.parquet: 170,089 rows, 9 variables
hfcs_2010_h_imp2.parquet: 68,627 rows, 36 variables
hfcs_2010_d_imp2.parquet: 68,627 rows, 164 variables
hfcs_2010_p_imp2.parquet: 170,089 rows, 9 variables
hfcs_2010_h_imp3.parquet: 68,627 rows, 36 variables
hfcs_2010_d_imp3.parquet: 68,627 rows, 164 variables
hfcs_2010_p_imp3.parquet: 170,089 rows, 9 variables
hfcs_2010_h_imp4.parquet: 68,627 rows, 36 variables
hfcs_2010_d_imp4.parquet: 68,627 rows, 164 variables
hfcs_2010_p_imp4.parquet: 170,089 rows, 9 variables
hfcs_2010_h_imp5.parquet: 68,627 rows, 36 variables
hfcs_2010_d_imp5.parquet: 68,627 rows, 164 variables
hfcs_2010_p_imp5.parquet: 170,089 rows, 9 variables
hfcs_2014_h_imp1.parquet: 84,611 rows, 40 variables
hfcs_2014_d_imp1.parquet: 84,611 rows, 170 variables
hfcs_2014_p_imp1.parquet: 210,346 rows, 9 variables
hfcs_2014_h_imp2.parquet: 84,611 rows, 40 variables
hfcs_2

,wave,table,raw_variables
0,2010,d,164
1,2010,h,36
2,2010,p,9
3,2014,d,170
4,2014,h,40
5,2014,p,9
6,2017,d,177
7,2017,h,41
8,2017,p,9
9,2021,d,175


## Stata labels

Save the variable descriptions and category labels from the source files. Missing value labels may also be defined in the release-specific Stata `.do` files; the label audit shows what was recovered directly from `.dta`.


In [8]:
# The Stata reader returns dictionaries keyed by variable name.
variable_rows, value_rows, label_audit_rows = [], [], []
for wave in WAVE_FOLDERS:
    for table, patterns in VARIABLE_PATTERNS.items():
        for imp in range(1, 6):
            source = raw_path(wave, table, imp)
            with StataReader(source) as reader:
                variable_labels = reader.variable_labels()
                value_labels = reader.value_labels()
            selected = select_variables(variable_labels, patterns)
            for var in selected:
                variable_rows.append({
                    "wave": wave, "table": table, "imputation": imp,
                    "variable": var, "variable_label": variable_labels.get(var, ""),
                })
                for number, label in value_labels.get(var, {}).items():
                    value_rows.append({
                        "wave": wave, "table": table, "imputation": imp,
                        "variable": var, "code": number, "value_label": label,
                    })
            label_audit_rows.append({
                "wave": wave, "table": table, "imputation": imp,
                "selected_variables": len(selected),
                "variables_with_labels": sum(bool(variable_labels.get(v)) for v in selected),
                "variables_with_value_labels": sum(bool(value_labels.get(v)) for v in selected),
            })

metadata_dir = OUTPUT_DIR / "metadata"
metadata_dir.mkdir(exist_ok=True)
pd.DataFrame(variable_rows).to_csv(metadata_dir / "variable_labels.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(value_rows, columns=["wave", "table", "imputation", "variable", "code", "value_label"]).to_csv(
    metadata_dir / "value_labels.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(label_audit_rows).to_csv(metadata_dir / "label_audit.csv", index=False, encoding="utf-8-sig")
print("Stata labels:", metadata_dir)
display(pd.DataFrame(label_audit_rows).groupby(["wave", "table"], as_index=False)["variables_with_value_labels"].max())


Stata labels: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\HFCS Data Download\HFCS_selected_raw\metadata


,wave,table,variables_with_value_labels
0,2010,d,0
1,2010,h,0
2,2010,p,0
3,2014,d,0
4,2014,h,0
5,2014,p,0
6,2017,d,0
7,2017,h,0
8,2017,p,0
9,2021,d,0
